### Notebook onde será feito a construção das features para aplicação no ML model em treinamento

com base no notebook EDA2 foram feitas análises e algumas definições de features, lags e janelas moveis que serão aplicadas no modelo.

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

# Set option to display all rows
# pd.set_option('display.max_rows', None)

In [2]:
df = pd.read_parquet(r"..\data\anp_enriched.parquet")

In [3]:
df.head()

,Valor Venda Dolar,Valor Compra Dolar,Close_BZ=F,High_BZ=F,Low_BZ=F,Open_BZ=F,Volume_BZ=F,ipca,pib_mensal,selic,...,Numero Rua,Complemento,Bairro,Cep,Produto,Data da Coleta,Valor de Venda,Valor de Compra,Unidade de Medida,Bandeira
2007-07-30,1.8809,1.8801,75.739998,76.529999,75.440002,75.849998,2575,0.24,89.38261,0.042956,...,S/N,None,CENTRO ATALAIAVAL,00000-000,GLP,2007-07-30,35.0,27.954545,R$ / 13 kg,BRANCA
2007-07-30,1.8809,1.8801,75.739998,76.529999,75.440002,75.849998,2575,0.24,89.38261,0.042956,...,S/N,None,JACUPEMBA,00000-000,GLP,2007-07-30,35.0,24.800000,R$ / 13 kg,BRANCA
2007-07-30,1.8809,1.8801,75.739998,76.529999,75.440002,75.849998,2575,0.24,89.38261,0.042956,...,S/N,None,,00000-000,GLP,2007-07-30,33.0,28.877037,R$ / 13 kg,BRANCA
2007-07-30,1.8809,1.8801,75.739998,76.529999,75.440002,75.849998,2575,0.24,89.38261,0.042956,...,S/N,None,TRES BARRAS,00000-000,GLP,2007-07-30,32.0,25.584923,R$ / 13 kg,BRANCA
2007-07-30,1.8809,1.8801,75.739998,76.529999,75.440002,75.849998,2575,0.24,89.38261,0.042956,...,S/N,None,,00000-000,GLP,2007-07-30,33.0,24.000000,R$ / 13 kg,BRANCA


## 1. Limpeza, manter somente columns necessárias

In [4]:
columns_utils = ['Produto', 'Valor de Venda','Data da Coleta','Estado - Sigla', 'Bandeira', 
                 'Valor Venda Dolar', 'Valor Compra Dolar', 'Close_BZ=F', 'High_BZ=F',
                'Low_BZ=F', 'Open_BZ=F', 'Volume_BZ=F', 'ipca', 'pib_mensal', 'selic',]
df = df[columns_utils]

# Filtrando produtos que iremos prever
df = df[df['Produto'].isin(['GASOLINA','GLP','ETANOL','DIESEL'])]


## 2. Aplicando column de data na frequencia semanal, afinal a previsao será semanal

In [5]:
df['dt_week'] = df['Data da Coleta'].dt.to_period('W')

df = df.dropna(how='any')

In [6]:
df.head()

,Produto,Valor de Venda,Data da Coleta,Estado - Sigla,Bandeira,Valor Venda Dolar,Valor Compra Dolar,Close_BZ=F,High_BZ=F,Low_BZ=F,Open_BZ=F,Volume_BZ=F,ipca,pib_mensal,selic,dt_week
2007-07-30,GLP,35.0,2007-07-30,SE,BRANCA,1.8809,1.8801,75.739998,76.529999,75.440002,75.849998,2575,0.24,89.38261,0.042956,2007-07-30/2007-08-05
2007-07-30,GLP,35.0,2007-07-30,ES,BRANCA,1.8809,1.8801,75.739998,76.529999,75.440002,75.849998,2575,0.24,89.38261,0.042956,2007-07-30/2007-08-05
2007-07-30,GLP,33.0,2007-07-30,SP,BRANCA,1.8809,1.8801,75.739998,76.529999,75.440002,75.849998,2575,0.24,89.38261,0.042956,2007-07-30/2007-08-05
2007-07-30,GLP,32.0,2007-07-30,MG,BRANCA,1.8809,1.8801,75.739998,76.529999,75.440002,75.849998,2575,0.24,89.38261,0.042956,2007-07-30/2007-08-05
2007-07-30,GLP,33.0,2007-07-30,MG,BRANCA,1.8809,1.8801,75.739998,76.529999,75.440002,75.849998,2575,0.24,89.38261,0.042956,2007-07-30/2007-08-05


será necessário aplicar o metodo ffill, combinado por estado x produto, pois se for aplicado diretamente no df o pandas nao sabe que cada estado × produto tem sua própria linha do tempo.

In [7]:
df_grouped = (
    df.groupby(["dt_week", "Estado - Sigla", "Produto"], as_index=False)
    ['Valor de Venda']
    .agg(
        price_sale_median='median',
        price_sale_std='std',
        n_postos='count'   # quantos postos reportaram naquela semana
    )
)

In [8]:
complete_date = pd.period_range(start=df['dt_week'].min(), end=df['dt_week'].max(), freq='W-SUN')

In [9]:
estados = df_grouped['Estado - Sigla'].unique()
produtos = df_grouped['Produto'].unique()

In [10]:
grade_completa = pd.MultiIndex.from_product(
    [complete_date, estados, produtos],
    names=['dt_week', 'Estado - Sigla', 'Produto']
).to_frame().reset_index(drop=True)

In [11]:
df_reconstruido = pd.merge(
    grade_completa, 
    df_grouped, 
    on=['dt_week', 'Estado - Sigla', 'Produto'], 
    how='left'
)

In [12]:
# Ordenar para garantir a sequência temporal correta dentro de cada grupo
df_reconstruido = df_reconstruido.sort_values(['Estado - Sigla', 'Produto', 'dt_week']).reset_index(drop=True)

# aplicando ffill nas columns com nan
df_reconstruido['price_sale_median'] = (
    df_reconstruido
    .groupby(['Estado - Sigla', 'Produto'])['price_sale_median']
    .ffill()
)

# em casos de n_postos = 1, o preco_std sera NaN, por nao ter registros naquele periodo semanal especifico, logo será preenchido por 0
df_reconstruido["price_sale_std"] = df_reconstruido["price_sale_std"].fillna(0)

# flag para o modelo conseguir diferenciar "dispersão genuinamente zero" de "dispersão não calculável por baixa amostra", vale criar uma feature binária
df_reconstruido['baixa_amostra'] = (df_reconstruido['n_postos'] <= 1).astype(int)

# n_postos recebe ffill (portanto reflete última cobertura conhecida)
df_reconstruido['n_postos'] = (
    df_reconstruido.groupby(['Estado - Sigla', 'Produto'])['n_postos']
    .ffill()
)

In [13]:
df_reconstruido.isna().sum()

dt_week                 0
Estado - Sigla          0
Produto                 0
price_sale_median    1786
price_sale_std          0
n_postos             1786
baixa_amostra           0
dtype: int64

#### Validações para exclusão dos dados no periodo de 2007 a 2008

In [14]:
values_nan = df_reconstruido[df_reconstruido["price_sale_median"].isna()]

In [15]:
values_nan.groupby(["dt_week", "Estado - Sigla", "Produto"]).count()

price_sale_median  \
dt_week               Estado - Sigla Produto                       
2007-07-30/2007-08-05 AC             DIESEL                    0   
                                     ETANOL                    0   
                                     GASOLINA                  0   
                      AL             DIESEL                    0   
                                     ETANOL                    0   
...                                                          ...   
2007-12-24/2007-12-30 TO             GASOLINA                  0   
2007-12-31/2008-01-06 RR             ETANOL                    0   
                      SE             DIESEL                    0   
                                     ETANOL                    0   
                                     GASOLINA                  0   

                                               price_sale_std  n_postos  \
dt_week               Estado - Sigla Produto                              
2007-07-30/2007-08-05 AC             DIESEL                 1         0   
                                     ETANOL                 1         0   
                                     GASOLINA               1         0   
                      AL             DIESEL                 1         0   
                                     ETANOL                 1         0   
...                                                       ...       ...   
2007-12-24/2007-12-30 TO             GASOLINA               1         0   
2007-12-31/2008-01-06 RR             ETANOL                 1         0   
                      SE             DIESEL                 1         0   
                                     ETANOL                 1         0   
                                     GASOLINA               1         0   

                                               baixa_amostra  
dt_week               Estado - Sigla Produto                  
2007-07-30/2007-08-05 AC             DIESEL                1  
                                     ETANOL                1  
                                     GASOLINA              1  
                      AL             DIESEL                1  
                                     ETANOL                1  
...                                                      ...  
2007-12-24/2007-12-30 TO             GASOLINA              1  
2007-12-31/2008-01-06 RR             ETANOL                1  
                      SE             DIESEL                1  
                                     ETANOL                1  
                                     GASOLINA              1  

[1786 rows x 4 columns]

In [16]:
values_nan["dt_week"].unique()

<PeriodArray>
['2007-07-30/2007-08-05', '2007-08-06/2007-08-12', '2007-08-13/2007-08-19',
 '2007-08-20/2007-08-26', '2007-08-27/2007-09-02', '2007-09-03/2007-09-09',
 '2007-09-10/2007-09-16', '2007-09-17/2007-09-23', '2007-09-24/2007-09-30',
 '2007-10-01/2007-10-07', '2007-10-08/2007-10-14', '2007-10-15/2007-10-21',
 '2007-10-22/2007-10-28', '2007-10-29/2007-11-04', '2007-11-05/2007-11-11',
 '2007-11-12/2007-11-18', '2007-11-19/2007-11-25', '2007-11-26/2007-12-02',
 '2007-12-03/2007-12-09', '2007-12-10/2007-12-16', '2007-12-17/2007-12-23',
 '2007-12-24/2007-12-30', '2007-12-31/2008-01-06']
Length: 23, dtype: period[W-SUN]

#### Filtrando para pegar dados após 2008-01-06

In [17]:
df_reconstruido = df_reconstruido[df_reconstruido["dt_week"] > '2007-12-31/2008-01-06']

In [18]:
df_reconstruido.isna().sum()

dt_week              0
Estado - Sigla       0
Produto              0
price_sale_median    0
price_sale_std       0
n_postos             0
baixa_amostra        0
dtype: int64

In [19]:
# APLICAR VARS EXOGENAS NO DF_RECONSTRUIDO
df_reconstruido.head()

,dt_week,Estado - Sigla,Produto,price_sale_median,price_sale_std,n_postos,baixa_amostra
23,2008-01-07/2008-01-13,AC,DIESEL,2.2,0.158598,47.0,0
24,2008-01-14/2008-01-20,AC,DIESEL,2.2,0.164542,49.0,0
25,2008-01-21/2008-01-27,AC,DIESEL,2.2,0.122404,35.0,0
26,2008-01-28/2008-02-03,AC,DIESEL,2.2,0.164139,49.0,0
27,2008-02-04/2008-02-10,AC,DIESEL,2.2,0.164950,49.0,0


### 2.1. Aplicando lags no df_reconstruido

In [20]:
# lags selecionados de acordo com os estudos feitos no notebook: EDA2
lags = [1, 2, 3, 4, 8, 27]

for lag in lags:
    # Criando lags para o price_sale_median (valor de venda -> mediana por estado, nao por postos... por isso há o std)
    df_reconstruido[f'price_sale_median_lag_{lag}'] = (
        df_reconstruido
        .groupby(['Estado - Sigla', 'Produto'])['price_sale_median']
        .shift(lag)
    )

#### 2.2. Aplicando rolling_mean(janelas movéis)

In [21]:
janelas = [2, 4, 8, 12]

for janela in janelas:
    df_reconstruido[f'price_sale_median_rolling_mean_{janela}'] = (
        df_reconstruido
        .groupby(['Estado - Sigla', 'Produto'])['price_sale_median']
        .transform(lambda x: x.shift(1).rolling(window=janela).mean())
    )

    # captura a volatilidade recente do preço — diferente do price_sale_std que já foi construido,
    #  que mede dispersão entre postos na mesma semana. 
    # Esse novo rolling_std mede dispersão ao longo do tempo, outra dimensão de informação.
    df_reconstruido[f'rolling_std_{janela}'] = (
        df_reconstruido
        .groupby(['Estado - Sigla', 'Produto'])['price_sale_median']
        .transform(lambda x: x.shift(1).rolling(window=janela).std())
    )

In [22]:
df_reconstruido = df_reconstruido.dropna()


### 3. Merge das variaveis exogenas com o df com features

In [23]:
columns_exogenas = [
    'dt_week', 
    'Valor Venda Dolar', 'Valor Compra Dolar', 
    'Close_BZ=F', 'High_BZ=F', 'Low_BZ=F', 'Open_BZ=F', 'Volume_BZ=F', 
    'ipca', 'pib_mensal', 'selic'
]

df_exogenas = df[columns_exogenas].drop_duplicates(subset=['dt_week'])

In [24]:
df_final = pd.merge(
    df_reconstruido, 
    df_exogenas, 
    on='dt_week', 
    how='left'
)

analisando alguns gaps após o merge

In [25]:
df_final.isna().sum()

dt_week                                 0
Estado - Sigla                          0
Produto                                 0
price_sale_median                       0
price_sale_std                          0
n_postos                                0
baixa_amostra                           0
price_sale_median_lag_1                 0
price_sale_median_lag_2                 0
price_sale_median_lag_3                 0
price_sale_median_lag_4                 0
price_sale_median_lag_8                 0
price_sale_median_lag_27                0
price_sale_median_rolling_mean_2        0
rolling_std_2                           0
price_sale_median_rolling_mean_4        0
rolling_std_4                           0
price_sale_median_rolling_mean_8        0
rolling_std_8                           0
price_sale_median_rolling_mean_12       0
rolling_std_12                          0
Valor Venda Dolar                    1512
Valor Compra Dolar                   1512
Close_BZ=F                        

In [26]:
df_final[df_final["ipca"].isna()]["dt_week"].value_counts()

dt_week
2009-04-20/2009-04-26    108
2009-04-27/2009-05-03    108
2009-08-17/2009-08-23    108
2009-08-24/2009-08-30    108
2015-08-17/2015-08-23    108
2020-08-24/2020-08-30    108
2020-08-31/2020-09-06    108
2020-09-07/2020-09-13    108
2020-09-14/2020-09-20    108
2020-09-21/2020-09-27    108
2020-09-28/2020-10-04    108
2020-10-05/2020-10-11    108
2020-10-12/2020-10-18    108
2022-09-19/2022-09-25    108
Freq: W-SUN, Name: count, dtype: int64

dropando os gaps

In [27]:
print(f"Shape antes do drop: {df_final.shape}")

df_final = df_final.dropna(subset=[
    'Valor Venda Dolar', 'Valor Compra Dolar',
    'Close_BZ=F', 'High_BZ=F', 'Low_BZ=F', 'Open_BZ=F', 'Volume_BZ=F',
    'ipca', 'pib_mensal', 'selic'
]).reset_index(drop=True)

print(f"Shape depois do drop: {df_final.shape}")

Shape antes do drop: (101736, 31)
Shape depois do drop: (100224, 31)


In [28]:
df_final.isna().sum()

dt_week                              0
Estado - Sigla                       0
Produto                              0
price_sale_median                    0
price_sale_std                       0
n_postos                             0
baixa_amostra                        0
price_sale_median_lag_1              0
price_sale_median_lag_2              0
price_sale_median_lag_3              0
price_sale_median_lag_4              0
price_sale_median_lag_8              0
price_sale_median_lag_27             0
price_sale_median_rolling_mean_2     0
rolling_std_2                        0
price_sale_median_rolling_mean_4     0
rolling_std_4                        0
price_sale_median_rolling_mean_8     0
rolling_std_8                        0
price_sale_median_rolling_mean_12    0
rolling_std_12                       0
Valor Venda Dolar                    0
Valor Compra Dolar                   0
Close_BZ=F                           0
High_BZ=F                            0
Low_BZ=F                 

##### Ultimas validações referente as janelas (rolling_mean)

decisao - aplicar janelas de 2, 4, 8 e 12 semanas, e fazer com que o feature importance do modelo diga, se é uma boa ou nao (em relação a feature)

In [29]:
from scipy import stats

# Testa várias janelas, não só 4 e 8
janelas_teste = [2, 4, 6, 8, 12, 16]

resultados = []

for produto in ['GASOLINA', 'ETANOL', 'DIESEL', 'GLP']:
    dados = df_final[df_final['Produto'] == produto].sort_values(['Estado - Sigla', 'dt_week'])
    
    for janela in janelas_teste:
        rolling = (
            dados.groupby('Estado - Sigla')['price_sale_median']
            .transform(lambda x: x.shift(1).rolling(window=janela).mean())
        )
        
        # Correlação entre a rolling mean e o preço da semana seguinte
        preco_futuro = dados.groupby('Estado - Sigla')['price_sale_median'].shift(-1)
        
        validos = rolling.notna() & preco_futuro.notna()
        rho, p = stats.spearmanr(rolling[validos], preco_futuro[validos])
        
        resultados.append({
            'produto': produto,
            'janela': janela,
            'rho': rho,
            'pvalue': p
        })

df_janelas = pd.DataFrame(resultados)
pivot = df_janelas.pivot(index='janela', columns='produto', values='rho')
print(pivot)

produto    DIESEL    ETANOL  GASOLINA       GLP
janela                                         
2        0.997273  0.996372  0.996639  0.997466
4        0.996486  0.995226  0.995770  0.997317
6        0.995558  0.993945  0.994731  0.997028
8        0.994565  0.992604  0.993605  0.996700
12       0.992401  0.989717  0.991216  0.996032
16       0.990304  0.986707  0.988849  0.995334


In [30]:
for janela in janelas_teste:
    rolling = (
        df_final.groupby(['Estado - Sigla', 'Produto'])['price_sale_median']
        .transform(lambda x: x.shift(1).rolling(window=janela).mean())
    )
    preco_real = df_final['price_sale_median']
    
    mae = (rolling - preco_real).abs().mean()
    print(f"Janela {janela}: MAE = {mae:.4f}")

Janela 2: MAE = 0.2241
Janela 4: MAE = 0.2639
Janela 6: MAE = 0.3039
Janela 8: MAE = 0.3432
Janela 12: MAE = 0.4197
Janela 16: MAE = 0.4936


In [31]:
df_final.columns

Index(['dt_week', 'Estado - Sigla', 'Produto', 'price_sale_median',
       'price_sale_std', 'n_postos', 'baixa_amostra',
       'price_sale_median_lag_1', 'price_sale_median_lag_2',
       'price_sale_median_lag_3', 'price_sale_median_lag_4',
       'price_sale_median_lag_8', 'price_sale_median_lag_27',
       'price_sale_median_rolling_mean_2', 'rolling_std_2',
       'price_sale_median_rolling_mean_4', 'rolling_std_4',
       'price_sale_median_rolling_mean_8', 'rolling_std_8',
       'price_sale_median_rolling_mean_12', 'rolling_std_12',
       'Valor Venda Dolar', 'Valor Compra Dolar', 'Close_BZ=F', 'High_BZ=F',
       'Low_BZ=F', 'Open_BZ=F', 'Volume_BZ=F', 'ipca', 'pib_mensal', 'selic'],
      dtype='object')

In [32]:
mem_bytes = df_final.memory_usage(deep=True).sum()
mem_mb = mem_bytes / (1024 ** 2)
print(f"Uso de memória: {mem_mb:.2f} MB")


Uso de memória: 31.90 MB


In [36]:
df_final.to_parquet(f"..\data\dados_anp_modelado.parquet")

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
C:\Users\ferna\AppData\Local\Temp\ipykernel_31176\1223868656.py:1: SyntaxWarning: invalid escape sequence '\d'
  df_final.to_parquet(f"..\data\dados_anp_modelado.parquet")
